In [ ]:
# 1. 单独安装 generate-chroma 缺少的其他辅助包（如 nglview）
!pip install nglview

# 2. 强制安装 generate-chroma，跳过其报错的旧依赖，直接使用 Colab 自带的新版本依赖
!pip install generate-chroma --no-deps

In [3]:
import torch
from chroma import Chroma, Protein, conditioners, api

# 检查设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ⚠️ 注意：加载模型权重需要 API Key
api.register_key("bc1b991a6d7d472896346e04e7b136a1") 

# 初始化 Chroma 模型
chroma_model = Chroma()

/usr/local/lib/python3.13/dist-packages/chroma/layers/graph.py:27: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Data saved to /tmp/chroma_weights/90e339502ae6b372797414167ce5a632/weights.pt
Computing reference stats for 2g3n
Data saved to /tmp/chroma_weights/03a3a9af343ae74998768a2711c8b7ce/weights.pt
Loaded from cache


In [4]:
import torch

# --- 兼容性补丁：解除 PyTorch 2.6 的 weights_only 安全限制 ---
original_load = torch.load
def patched_load(*args, **kwargs):
    # 强制将 weights_only 设置为 False，允许加载旧版模型的 numpy 数据
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load
# --------------------------------------------------------

# 然后再正常导入和初始化 Chroma
from chroma import Chroma, Protein, conditioners, api

# 下面继续写你原来的代码...
# api.register_key("...")
# chroma = Chroma()

In [7]:
"""
Official Chroma Colab notebook, distilled into a runnable stage-2 / stage-3 demo.

This file follows the public ChromaAPI.ipynb notebook from the generatebio/chroma
repository and keeps the original demo parameters as closely as possible.

Usage in Colab:
1. Install generate-chroma first if needed.
2. Run the API key registration step.
3. Execute the demo blocks below.

Usage in a local GPU environment:
1. pip install generate-chroma
2. Set your Chroma API key when prompted.
3. Run the script.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import torch

from chroma import Chroma, Protein, conditioners, api


try:
    from IPython.display import display as ipy_display
except Exception:  # pragma: no cover - plain Python fallback
    ipy_display = None


def show(obj: Any) -> None:
    if ipy_display is not None:
        ipy_display(obj)
    else:
        print(obj)


def save_sample(protein: Protein, path: str) -> Protein:
    protein.to(path)
    return Protein(path)


def main() -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    api.register_key(input("Enter API key: "))

    # Initialize the model.
    chroma = Chroma()

    # ---------------------------------------------------------------------
    # Stage 2: core demo blocks from the official notebook
    # ---------------------------------------------------------------------

    # Unconditional sampling.
    torch.manual_seed(0)
    protein = chroma.sample()
    print(protein)
    protein = save_sample(protein, "chroma_sample.cif")
    show(protein)
    elbo = chroma.score(protein)["elbo"].score
    print(f"sample elbo: {elbo}")

    # Symmetry conditioning.
    SYMMETRY_GROUP = "C_7"
    SUBUNIT_SIZES = [100]
    KNBR = 2

    torch.manual_seed(0)
    symmetry_conditioner = conditioners.SymmetryConditioner(
        G=SYMMETRY_GROUP,
        num_chain_neighbors=KNBR,
    )
    symmetric_protein = chroma.sample(
        chain_lengths=SUBUNIT_SIZES,
        conditioner=symmetry_conditioner,
        langevin_factor=8,
        inverse_temperature=8,
        sde_func="langevin",
        potts_symmetry_order=symmetry_conditioner.potts_symmetry_order,
    )
    show(symmetric_protein)

    # Infilling / substructure conditioning.
    TESTED_PDBS = ["3bdi", "5sv5", "6qaz", "2e0q", "5xb0", "6bde", "1a8q", "5o0t", "1drf", "1shg"]
    MASK_PERCENT = 0.5
    PDB_ID = TESTED_PDBS[0]

    from chroma.utility.chroma import plane_split_protein

    protein = Protein(PDB_ID, canonicalize=True, device=device)
    X, C, _ = protein.to_XCS()
    residues_to_design = plane_split_protein(X, C, protein, MASK_PERCENT).nonzero()[:, 1].tolist()
    protein.sys.save_selection(gti=residues_to_design, selname="infilling_selection")

    infilling_conditioner = conditioners.SubstructureConditioner(
        protein,
        backbone_model=chroma.backbone_network,
        selection="namesel infilling_selection",
    ).to(device)

    torch.manual_seed(0)
    infilled_protein = chroma.sample(
        protein_init=protein,
        conditioner=infilling_conditioner,
        langevin_factor=4.0,
        langevin_isothermal=True,
        inverse_temperature=8.0,
        sde_func="langevin",
        steps=500,
    )
    show(infilled_protein)

    # Shape conditioning.
    LETTER = "G"
    NUM_RESIDUES = 1000

    from chroma.utility.chroma import letter_to_point_cloud

    letter_point_cloud = letter_to_point_cloud(LETTER)
    shape_conditioner = conditioners.ShapeConditioner(
        letter_point_cloud,
        chroma.backbone_network.noise_schedule,
        autoscale_num_residues=NUM_RESIDUES,
    ).to(device)

    torch.manual_seed(0)
    shaped_protein = chroma.sample(chain_lengths=[NUM_RESIDUES], conditioner=shape_conditioner)
    show(shaped_protein)

    # Protein class conditioning.
    CATH_ANNOTATION = "2.60.40"

    torch.manual_seed(0)
    cath_conditioner = conditioners.ProClassConditioner("cath", CATH_ANNOTATION)
    cath_conditioned_protein = chroma.sample(conditioner=cath_conditioner)
    show(cath_conditioned_protein)

    # Natural language conditioning.
    CAPTION = "Crystal structure of SH2 domain"

    torch.manual_seed(0)
    caption_conditioner = conditioners.ProCapConditioner(CAPTION, -1)
    caption_conditioned_protein = chroma.sample(
        steps=200,
        chain_lengths=[110],
        conditioner=caption_conditioner,
    )
    show(caption_conditioned_protein)

    # ---------------------------------------------------------------------
    # Stage 3: save / inspect / lightweight verification hooks
    # ---------------------------------------------------------------------

    out_dir = Path("chroma_demo_outputs")
    out_dir.mkdir(exist_ok=True)

    protein = save_sample(protein, str(out_dir / "chroma_sample.cif"))
    symmetric_protein.to(str(out_dir / "symmetric_c7.cif"))
    infilled_protein.to(str(out_dir / "infilling_demo.cif"))
    shaped_protein.to(str(out_dir / "shape_g.cif"))
    cath_conditioned_protein.to(str(out_dir / "cath_2.60.40.cif"))
    caption_conditioned_protein.to(str(out_dir / "caption_sh2_domain.cif"))

    print(f"Saved demo structures to: {out_dir.resolve()}")


if __name__ == "__main__":
    main()


Using device: cuda
Using cached data from /tmp/chroma_weights/90e339502ae6b372797414167ce5a632/weights.pt
Loaded from cache
Using cached data from /tmp/chroma_weights/03a3a9af343ae74998768a2711c8b7ce/weights.pt
Loaded from cache


Integrating SDE:   0%|          | 0/500 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/100 [00:00<?, ?it/s]

Protein: system
> Chain A (100 residues)
SAAAAAQRRRDEELAKKFEEILSEMKQPLTPEKVKKIAENFDEILELLKRWALLHPDRKDEVEEAFERLKSVFDQAGIGEEATKKIDELLEEVERELAAR




...

Integrating SDE:   0%|          | 0/200 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/110 [00:00<?, ?it/s]

NGLWidget()

Saved demo structures to: /content/chroma_demo_outputs
